# SensoDat: Computing Road Features
We establish first a connection to the database.

In [1]:
import os

from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

load_dotenv()

username = os.getenv('SENSODAT_USERNAME')
password = os.getenv('SENSODAT_PASSWORD')
host = os.getenv('SENSODAT_HOST')
port = os.getenv('SENSODAT_PORT')

uri = "mongodb://{}:{}@{}:{}".format(username, password, host, port)

client = MongoClient(uri, server_api=ServerApi('1'))
db = client.get_database('sdc_sim_data')

Then we create a sample query to check if everything works by counting the number of passing tests.

In [2]:
query_passing = {"OpenDRIVE.header.sdc_test_info.@test_outcome": "PASS"}
nr_passing_tests = db['campaign_2_frenetic'].count_documents(filter=query_passing)
print(nr_passing_tests)

562


The following cell depicts an example how we can query and aggregate data from SensoDat. Note: SensoDat uses MongoDB!

In [3]:
query_filter = {}
select = {"OpenDRIVE": 1, "execution_data.simulation_data": 1}

query_result = db['campaign_2_frenetic'].find(query_filter, select)
trace_data = None

print(query_result[0]['OpenDRIVE']['road'].keys())

tests_sim_data = [
    {'time_series_data': x['execution_data']['simulation_data'], 'meta_data': x['OpenDRIVE']['header']['sdc_test_info'],
     'road': x['OpenDRIVE']['road']} for x in query_result]
#tests_sim_data[0]

dict_keys(['@id', '@length', 'link', 'planView', 'lanes'])


## Road Features
Let us load some data and compute the road features.

In [4]:
def load_data(collection_string: str = 'campaign_2_frenetic'):
    stage_one = {'$project': {
        'road': {
            '$map': {
                'input': "$OpenDRIVE.road.planView.geometry",
                'as': "road_array",
                'in': "$$road_array",
            },
        },
        'simulation_data': "$execution_data.simulation_data",
        'meta_data': {
            'test_info': "$OpenDRIVE.header.sdc_test_info",
            'run_config': "$OpenDRIVE.header.run_config",
        },
    }}
    stage_two = {'$project': {
        'road_points': {
            '$map': {
                'input': "$road",
                'as': "road",
                'in': {
                    'x': {'$toDouble': "$$road.@x"},
                    'y': {'$toDouble': "$$road.@y"},
                },
            },
        },
        'simulation_data': "$simulation_data",
        'meta_data': {
            'test_info': {
                'test_id': "$meta_data.test_info.@test_id",
                'test_outcome': "$meta_data.test_info.@test_outcome",
                'test_duration': {'$toDouble': "$meta_data.test_info.@test_duration"},
                'is_valid': {'$toBool': "$meta_data.test_info.@is_valid"},
            },
            'run_config': {
                'rf': {'$toDouble': "$meta_data.run_config.@rf"},
                'oob': {'$toDouble': "$meta_data.run_config.@oob"},
                'max_speed': {'$toInt': "$meta_data.run_config.@max_speed"},
            }
        }
    }}
    pipeline = [stage_one, stage_two]

    return db[collection_string].aggregate(pipeline)





### Here we need SDC-Scissor: https://github.com/christianbirchler-org/sdc-scissor

In [5]:
from sdc_scissor.testing_api.test import Test
from sdc_scissor.feature_extraction_api.angle_based_strategy import AngleBasedStrategy
from sdc_scissor.feature_extraction_api.feature_extraction import FeatureExtractor
import pandas as pd


def create_feature_dataframe(collection_name: str):
    features_dict_ = {"full_road_diversity": [], "mean_road_diversity": [], "direct_distance": [], "max_angle": [],
                      "max_pivot_off": [], "mean_angle": [], "mean_pivot_off": [], "median_angle": [],
                      "median_pivot_off": [], "min_angle": [], "min_pivot_off": [], "num_l_turns": [],
                      "num_r_turns": [],
                      "num_straights": [], "road_distance": [], "std_angle": [], "std_pivot_off": [], "total_angle": [],
                      "test_outcome": []}

    test_cursor_ = load_data(collection_name)
    while test_cursor_.alive:
        test_dict_ = test_cursor_.next()
        test_id_ = test_dict_['meta_data']['test_info']['test_id']
        
        road_points = [[point['x'], point['y']] for point in test_dict_['road_points']]
        test_outcome = test_dict_['meta_data']['test_info']['test_outcome']

        test = Test(test_id=test_id_, road_points=road_points, test_outcome=test_outcome)
        test.interpolated_road_points = road_points

        segmentation_ = AngleBasedStrategy(angle_threshold=5, decision_distance=10)
        feature_extractor_ = FeatureExtractor(segmentation_strategy=segmentation_)

        features = feature_extractor_.extract_features(test)

        features_dict_['test_outcome'].append(test_outcome)
        features_dict_["full_road_diversity"].append(features.full_road_diversity)
        features_dict_["mean_road_diversity"].append(features.mean_road_diversity)
        features_dict_["direct_distance"].append(features.direct_distance)
        features_dict_["max_angle"].append(features.max_angle)
        features_dict_["max_pivot_off"].append(features.max_pivot_off)
        features_dict_["mean_angle"].append(features.mean_angle)
        features_dict_["mean_pivot_off"].append(features.mean_pivot_off)
        features_dict_["median_angle"].append(features.median_angle)
        features_dict_["median_pivot_off"].append(features.median_pivot_off)
        features_dict_["min_angle"].append(features.min_angle)
        features_dict_["min_pivot_off"].append(features.min_pivot_off)
        features_dict_["num_l_turns"].append(features.num_l_turns)
        features_dict_["num_r_turns"].append(features.num_r_turns)
        features_dict_["num_straights"].append(features.num_straights)
        features_dict_["road_distance"].append(features.road_distance)
        features_dict_["std_angle"].append(features.std_angle)
        features_dict_["std_pivot_off"].append(features.std_pivot_off)
        features_dict_["total_angle"].append(features.total_angle)

    return pd.DataFrame(features_dict_)

In [6]:
dd = create_feature_dataframe('campaign_2_frenetic')


In [7]:
dd

,full_road_diversity,mean_road_diversity,direct_distance,max_angle,max_pivot_off,mean_angle,mean_pivot_off,median_angle,median_pivot_off,min_angle,min_pivot_off,num_l_turns,num_r_turns,num_straights,road_distance,std_angle,std_pivot_off,total_angle,test_outcome
0,340.933606,24.352400,177.086153,22.366940,91.215740,-5.158289,42.863304,-4.570520,40.819524,-66.192381,0.0,6,7,1,218.465364,25.217026,22.357293,-72.216045,FAIL
1,280.322426,23.360202,83.868485,94.821544,87.470701,19.509384,35.842622,9.371390,26.165940,-12.753694,0.0,7,3,2,179.162657,31.657402,27.564391,234.112614,FAIL
2,236.743356,14.796460,173.986940,25.326800,100.805436,-4.017233,28.293299,3.293615,25.783945,-61.910118,0.0,6,4,6,228.529705,26.192604,30.176553,-64.275729,PASS
3,125.996590,9.692045,112.225486,48.996975,67.147521,2.853649,29.802570,4.353868,24.194448,-33.903666,0.0,6,5,2,169.308724,26.055513,21.086767,37.097440,PASS
4,218.431523,27.303940,109.580087,4.783974,41.526188,-22.576518,18.086506,-16.002425,17.770163,-70.880434,0.0,0,5,3,168.968734,28.584189,17.074607,-180.612146,FAIL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923,430.061360,61.437337,95.974737,104.878332,56.071591,-3.650788,27.425465,-21.610434,28.167283,-60.552586,0.0,2,4,1,149.355158,53.942418,17.091393,-25.555514,PASS
924,134.592599,9.613757,154.604234,36.915480,123.519817,-4.491734,31.825606,-3.440220,26.628839,-55.449254,0.0,3,6,5,198.650668,21.193356,34.925781,-62.884278,PASS
925,94.569709,5.910607,144.288079,318.853296,60.025165,-2.756022,22.182322,0.531136,22.801923,-340.716839,0.0,7,5,4,218.906103,122.292878,20.171083,-44.096351,FAIL
926,84.372701,5.624847,182.194232,25.924850,104.528415,-18.984497,28.611572,-0.808150,24.653844,-325.722708,0.0,6,4,5,218.379518,86.105500,30.331963,-284.767457,FAIL


# Compute road features till point of OOB

In [8]:
from shapely.geometry import box
import numpy as np

def out_of_bounds(pos: tuple, rm: dict, oob: float) -> bool:
    """
    Compute of the car at the current position is out of bounds (OOB) according to the OOB parameter.
    :param pos: The current position of the car
    :param rm: Dictionary containing road information
    :param oob: Out of bounds parameter
    :return: True if OOB, False otherwise
    """
    car_model = box(pos[0]-1, pos[1]-1, pos[0]+1, pos[1]+1)
    diff = car_model.difference(rm['right_lane'])
    actual_oob = diff.area / car_model.area
    return actual_oob > oob

def closest_point(pos: tuple, road_pts: list[tuple[float, float]]) -> (int, int):
    """
    This function is purely functional.
    :param pos: 
    :param road_pts: 
    :return: Return the index of the closest road point to the given position coordinates.
    """
    pos_np = np.array(pos)
    curr_min_dist = -1
    curr_min_dist_idx = -1
    for i, road_pt in enumerate(road_pts):
        road_pt_np = np.array(road_pt)
        norm = np.linalg.norm(road_pt_np - pos_np)
        if norm < curr_min_dist or curr_min_dist == -1:
            curr_min_dist = norm
            curr_min_dist_idx = i

    return curr_min_dist_idx, curr_min_dist


In [9]:

from sdc_scissor.feature_extraction_api.feature_extraction import RoadFeatures
from shapely.geometry import LineString

class NoOOBException(Exception):
    pass

def road_features_till_oob(sensodat_entry, oob=0.1) -> (RoadFeatures, str):
    """
    Return road features and test outcome according to the given oob.
    Only road features till OOB are considered. If there is no OOB then the road features of the whole road are extracted.
    :param sensodat_entry: 
    :param oob: 
    :return:  Road features and test outcome according to the given oob.
    """
    
    data = sensodat_entry
    road_points = [(point['x'], point['y']) for point in data['road_points']]
    trajectory = [(d['time'], d['position'][0], d['position'][1]) for d in data['simulation_data']]
    test_id = data['meta_data']['test_info']['test_id']

    segmentation = AngleBasedStrategy(angle_threshold=5, decision_distance=10)
    feature_extractor = FeatureExtractor(segmentation_strategy=segmentation)
    
    center_line: LineString = LineString(coordinates=road_points)
    right_lane = center_line.buffer(distance=-5, single_sided=True)
    ideal_trajectory: LineString = center_line.parallel_offset(distance=2.5)
    
    road_model = {
        'road_points': road_points,
        'center_line': center_line,
        'right_lane': right_lane,
        'ideal_trajectory': ideal_trajectory
    }
    
    
    for idx, tick_data in enumerate(trajectory):
        timestamp = tick_data[0]
        position = (tick_data[1], tick_data[2])
        if out_of_bounds(position, road_model, oob):
            index_of_road_point_with_OOB, min_dist = closest_point(position, road_points)
            test = Test(test_id=test_id, road_points=road_points[:index_of_road_point_with_OOB], test_outcome="FAIL")
            return feature_extractor.extract_features(test), "FAIL"
        if idx == len(trajectory) - 1:
            test = Test(test_id=test_id, road_points=road_points, test_outcome="PASS")
            return feature_extractor.extract_features(test), "PASS"


## Load SensoDat Collection
You can use any collection of SensoDat.

In [10]:
# use any collection of SensoDat
d = load_data('campaign_2_frenetic')

## Extract road features

In [11]:

features_dict = {"test_id": [], 'test_outcome': [], "full_road_diversity": [], "mean_road_diversity": [],
                 "direct_distance": [], "max_angle": [], "max_pivot_off": [], "mean_angle": [], "mean_pivot_off": [],
                 "median_angle": [], "median_pivot_off": [], "min_angle": [], "min_pivot_off": [], "num_l_turns": [],
                 "num_r_turns": [], "num_straights": [], "road_distance": [], "std_angle": [], "std_pivot_off": [],
                 "total_angle": []}

cnt_passing_test = 0
cnt_failing_test = 0
cnt_error = 0

OOB=0.3

for data in d:
    try:
        features, outcome = road_features_till_oob(data, oob=OOB)
        if outcome == "PASS":
            cnt_passing_test += 1
        elif outcome == "FAIL":
            cnt_failing_test += 1
        else:
            raise Exception()
        
        features_dict["test_id"].append(data['meta_data']['test_info']['test_id'])
        features_dict['test_outcome'].append(outcome)
        features_dict["full_road_diversity"].append(features.full_road_diversity)
        features_dict["mean_road_diversity"].append(features.mean_road_diversity)
        features_dict["direct_distance"].append(features.direct_distance)
        features_dict["max_angle"].append(features.max_angle)
        features_dict["max_pivot_off"].append(features.max_pivot_off)
        features_dict["mean_angle"].append(features.mean_angle)
        features_dict["mean_pivot_off"].append(features.mean_pivot_off)
        features_dict["median_angle"].append(features.median_angle)
        features_dict["median_pivot_off"].append(features.median_pivot_off)
        features_dict["min_angle"].append(features.min_angle)
        features_dict["min_pivot_off"].append(features.min_pivot_off)
        features_dict["num_l_turns"].append(features.num_l_turns)
        features_dict["num_r_turns"].append(features.num_r_turns)
        features_dict["num_straights"].append(features.num_straights)
        features_dict["road_distance"].append(features.road_distance)
        features_dict["std_angle"].append(features.std_angle)
        features_dict["std_pivot_off"].append(features.std_pivot_off)
        features_dict["total_angle"].append(features.total_angle)
        
    except Exception as e:
        print("Error while extracting feature: {}".format(e))
        cnt_error += 1


print('{} passing tests'.format(cnt_passing_test))
print('{} failing tests'.format(cnt_failing_test))
print('{} errors'.format(cnt_error))

features_till_oob_df =pd.DataFrame(features_dict)

Error while extracting feature: too many indices for array: array is 1-dimensional, but 2 were indexed
320 passing tests
607 failing tests
1 errors
